# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Access, print name and description from Croissant metadata as object attributes
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We print the available record sets and their associated fields, referencing by `@id`.

In [ ]:
# Get all record sets from metadata
record_sets = dataset.metadata.recordSet

if record_sets is None or len(record_sets) == 0:
    print("No record sets found in the dataset metadata.")
else:
    for rs in record_sets:
        # Each record set is an object with its own attributes
        print(f"Record set @id: {rs['@id']}")
        if 'field' in rs:
            for f in rs['field']:
                print(f"  Field @id: {f['@id']}, Name: {f.get('name', 'Unnamed field')}")
        else:
            print("  No fields found in this record set.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# --- Dynamically collect record set @id for extraction ---
record_set_ids = []

if record_sets is not None:
    for rs in record_sets:
        record_set_ids.append(rs['@id'])

# Load each record set as DataFrame
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set @id: {record_set_id}")
        print(df.columns.tolist())
        print(df.head())
    except Exception as e:
        print(f"Could not load records for record set @id: {record_set_id}")
        print(e)

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
Operations can include removing outliers, transforming data distributions, or grouping data by key attributes.

We demonstrate with the first available record set and its fields. All field references use their `@id`.

In [ ]:
# Find a numeric field in the first available record set
chosen_record_set_id = None
numeric_field_id = None

for rs in (record_sets or []):
    fields = rs.get('field', [])
    for field in fields:
        if field.get('dataType', '').lower() in ['integer', 'float', 'number']:
            numeric_field_id = field['@id']
            chosen_record_set_id = rs['@id']
            break
    if numeric_field_id:
        break

if chosen_record_set_id and numeric_field_id:
    df = dataframes[chosen_record_set_id]

    # Check if the numeric field exists in DataFrame columns
    if numeric_field_id in df.columns:
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the selected numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by a categorical field
        group_field_id = None
        for field in rs.get('field', []):
            if field.get('dataType', '').lower() == 'text':
                group_field_id = field['@id']
                break
        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print(f"Numeric field {numeric_field_id} not found in columns of DataFrame for record set {chosen_record_set_id}.")
else:
    print("Could not identify a numeric field for EDA in available record sets.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Simple visualization for numeric data
if chosen_record_set_id and numeric_field_id:
    df = dataframes[chosen_record_set_id]
    if numeric_field_id in df.columns:
        plt.hist(df[numeric_field_id].dropna(), bins=10, color='skyblue')
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()

        # If a group field exists, show group comparison
        if 'group_field_id' in locals() and group_field_id and group_field_id in df.columns:
            df.boxplot(column=numeric_field_id, by=group_field_id)
            plt.title(f"{numeric_field_id} by {group_field_id}")
            plt.suptitle("")
            plt.xlabel(group_field_id)
            plt.ylabel(numeric_field_id)
            plt.show()
    else:
        print("No numeric field found for visualization.")
else:
    print("No record set and numeric field found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded the dataset metadata and explored the available record sets and fields, explicitly referencing their `@id`s.
- Data from each record set was extracted and loaded into pandas DataFrames.
- We identified numeric and text fields for basic filtering, normalization, grouping, and visualization.
- This structured approach enables clear, reproducible FAIR dataset exploration using Croissant schemas and `mlcroissant`.

**Note:** Due to the dataset's small size and clinical specificity, traditional ML/AI modeling is not recommended, but the dataset is valuable for case analysis and further clinical research.